# 03 - Test the H2O Model with the Azure ML Inference Server

This notebook runs the registered-model scoring contract locally with `azureml-inference-server-http`. It generates the scoring script, starts one local inference worker, calls the health and scoring routes, validates predictions against the golden fixture, tests an invalid request, displays logs, and shuts everything down.

No Azure endpoint, deployment, compute, or networking resource is created.

## 1. Configure the Local Test

Use the repository `.venv`. Notebook 01 must have created the binary-model bundle under `tmp/h2o_binary/taxi_fare`. This notebook loads the same repository-root `.env` as the Azure notebooks, although this local test does not require Azure account values.

The local server uses:

- Port `5001` for health and scoring
- One inference worker
- One loopback-only H2O node on port `54321`
- H2O `3.46.0.12`

One worker is intentional: every worker would otherwise try to own the same local H2O port.

In [ ]:
from pathlib import Path
import importlib.metadata as package_metadata
import json
import os
import platform
import shutil
import socket
import subprocess
import sys
import time

import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

for folder in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (folder / "tmp" / "h2o_binary" / "taxi_fare" / "model_manifest.json").is_file():
        REPO_ROOT = folder
        break
else:
    raise FileNotFoundError("Run Notebook 01 before Notebook 03")

ENV_FILE = REPO_ROOT / ".env"
load_dotenv(ENV_FILE)

MODEL_DIR = REPO_ROOT / "tmp" / "h2o_binary" / "taxi_fare"
LOCAL_DIR = REPO_ROOT / "tmp" / "h2o_online_local"
LOCAL_DIR.mkdir(parents=True, exist_ok=True)
SCORE_PATH = LOCAL_DIR / "score.py"
REQUEST_PATH = LOCAL_DIR / "sample_request.json"
SERVER_LOG_PATH = LOCAL_DIR / "inference_server.log"
SCORING_PORT = 5001

INFERENCE_SERVER = Path(sys.executable).parent / "azmlinfsrv.exe"
if not INFERENCE_SERVER.is_file():
    candidate = shutil.which("azmlinfsrv")
    if not candidate:
        raise FileNotFoundError("Install azureml-inference-server-http==1.4.1 in .venv")
    INFERENCE_SERVER = Path(candidate)

if not shutil.which("java"):
    java_bins = sorted((Path.home() / ".jdk").glob("jdk-17*/bin"), reverse=True)
    if not java_bins:
        raise FileNotFoundError("Install JDK 17 before running the local server")
    os.environ["JAVA_HOME"] = str(java_bins[0].parent)
    os.environ["PATH"] = str(java_bins[0]) + os.pathsep + os.environ["PATH"]

versions = {
    name: package_metadata.version(name)
    for name in ["azureml-inference-server-http", "h2o", "pandas", "numpy"]
}
if versions["azureml-inference-server-http"] != "1.4.1":
    raise RuntimeError("Notebook 03 requires azureml-inference-server-http==1.4.1")
if versions["h2o"] != "3.46.0.12":
    raise RuntimeError("Notebook 03 requires h2o==3.46.0.12")

print(f"Python: {sys.version.split()[0]}")
print(f"Model:  {MODEL_DIR.relative_to(REPO_ROOT)}")
print(f"Server: {INFERENCE_SERVER.name}")
versions

## 2. Generate the Azure ML Scoring Script

Azure ML calls `init()` once when the inference worker starts and calls `run()` for every request. The model is loaded once, not once per prediction.

The scoring script validates the artifact before H2O starts and never logs request feature values.

In [ ]:
SCORE_SOURCE = '''
import atexit
import hashlib
import json
import logging
import os
import threading
from pathlib import Path

import h2o
import pandas as pd
from azureml_inference_server_http.api.aml_response import AMLResponse

_model = None
_manifest = None
_predict_lock = threading.Lock()


def _sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _find_manifest(model_root):
    matches = list(model_root.rglob("model_manifest.json"))
    if len(matches) != 1:
        raise RuntimeError(f"Expected one model_manifest.json, found {len(matches)}")
    return matches[0]


def _predict(frame):
    h2o_frame = None
    prediction_frame = None
    try:
        h2o_frame = h2o.H2OFrame(frame)
        for column in _manifest.get("categorical_features", []):
            h2o_frame[column] = h2o_frame[column].asfactor()
        prediction_frame = _model.predict(h2o_frame)
        return prediction_frame.as_data_frame()["predict"].astype(float).tolist()
    finally:
        if prediction_frame is not None:
            h2o.remove(prediction_frame)
        if h2o_frame is not None:
            h2o.remove(h2o_frame)


def _parse_request(raw_data):
    payload = json.loads(raw_data) if isinstance(raw_data, (str, bytes)) else raw_data
    input_data = payload.get("input_data") if isinstance(payload, dict) else None
    if not isinstance(input_data, dict):
        raise ValueError("Request must contain an input_data object")

    columns = input_data.get("columns")
    rows = input_data.get("data")
    expected_columns = _manifest["features"]
    if columns != expected_columns:
        raise ValueError(f"Expected columns in this order: {expected_columns}")
    if not isinstance(rows, list) or not 1 <= len(rows) <= 100:
        raise ValueError("Request must contain between 1 and 100 rows")
    if any(not isinstance(row, list) or len(row) != len(columns) for row in rows):
        raise ValueError("Every row must have one value for each column")

    frame = pd.DataFrame(rows, columns=columns)
    for column in expected_columns:
        frame[column] = pd.to_numeric(frame[column], errors="raise")
    if frame.isna().any().any():
        raise ValueError("Null values are not accepted by this endpoint contract")
    return frame


def _shutdown_h2o():
    try:
        if h2o.connection() is not None:
            h2o.cluster().shutdown(prompt=False)
    except Exception:
        logging.exception("H2O shutdown failed")


def init():
    global _model, _manifest

    model_root = Path(os.environ["AZUREML_MODEL_DIR"]).resolve()
    manifest_path = _find_manifest(model_root)
    _manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    model_path = manifest_path.parent / _manifest["model_file"]

    if _manifest.get("model_format") != "h2o_binary":
        raise RuntimeError("The registered asset is not an H2O binary model")
    if h2o.__version__ != _manifest["h2o_version"]:
        raise RuntimeError(
            f"Expected h2o=={_manifest['h2o_version']}, found {h2o.__version__}"
        )
    if _sha256(model_path) != _manifest["files"][model_path.name]:
        raise RuntimeError("Binary model checksum does not match the manifest")

    h2o.no_progress()
    h2o.init(
        ip="127.0.0.1",
        port=54321,
        start_h2o=True,
        nthreads=1,
        max_mem_size="2G",
        strict_version_check=True,
        bind_to_localhost=True,
        verbose=False,
        telemetry=False,
    )
    _model = h2o.load_model(str(model_path))
    warmup = pd.read_csv(manifest_path.parent / "golden_input.csv").head(1)
    with _predict_lock:
        _predict(warmup)
    atexit.register(_shutdown_h2o)
    logging.info(
        "H2O model initialized: name=%s version=%s h2o=%s",
        _manifest["model_name"],
        _manifest["model_version"],
        _manifest["h2o_version"],
    )


def run(raw_data):
    try:
        frame = _parse_request(raw_data)
    except (ValueError, TypeError, KeyError, json.JSONDecodeError) as exc:
        return AMLResponse({"error": str(exc)}, 400, json_str=True)

    with _predict_lock:
        predictions = _predict(frame)

    return {
        "predictions": predictions,
        "model_name": _manifest["model_name"],
        "model_version": _manifest["model_version"],
        "h2o_version": _manifest["h2o_version"],
    }
'''.lstrip()

SCORE_PATH.write_text(SCORE_SOURCE, encoding="utf-8")
compile(SCORE_SOURCE, str(SCORE_PATH), "exec")
print(f"Generated scoring script: {SCORE_PATH}")

## 3. Create a Golden Request

The request uses Azure ML's tabular `input_data` contract. All 20 golden rows are sent in one request, which is appropriate for a local functional test rather than a load test.

In [ ]:
manifest = json.loads((MODEL_DIR / "model_manifest.json").read_text(encoding="utf-8"))
golden_input = pd.read_csv(MODEL_DIR / "golden_input.csv")
golden_expected = pd.read_csv(MODEL_DIR / "golden_expected.csv")

request_payload = {
    "input_data": {
        "columns": manifest["features"],
        "data": golden_input[manifest["features"]].values.tolist(),
    }
}
REQUEST_PATH.write_text(json.dumps(request_payload, indent=2), encoding="utf-8")

print(f"Request rows: {len(request_payload['input_data']['data'])}")
print(f"Request file: {REQUEST_PATH}")
request_payload["input_data"]["data"][:2]

## 4. Start the Local Inference Server

This is the same inference server Azure ML uses for a custom scoring script. Startup is complete only when `init()` has started H2O, verified and loaded the binary model, and completed one warm-up prediction.

In [ ]:
def port_is_open(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as connection:
        connection.settimeout(0.25)
        return connection.connect_ex(("127.0.0.1", port)) == 0


def stop_local_server():
    process = globals().get("server_process")
    log_handle = globals().get("server_log_handle")
    if process is not None and process.poll() is None:
        if platform.system() == "Windows":
            subprocess.run(
                ["taskkill", "/PID", str(process.pid), "/T", "/F"],
                capture_output=True,
                check=False,
            )
        else:
            process.terminate()
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            process.kill()
    if log_handle is not None and not log_handle.closed:
        log_handle.close()


stop_local_server()
if port_is_open(SCORING_PORT) or port_is_open(54321):
    raise RuntimeError("Ports 5001 and 54321 must be free before startup")

server_log_handle = SERVER_LOG_PATH.open("w", encoding="utf-8")
server_environment = os.environ.copy()
for inherited_key in ("AZUREML_ENTRY_SCRIPT", "AZUREML_MODEL_DIR", "WORKSPACE_NAME"):
    server_environment.pop(inherited_key, None)
server_environment["PYTHONUNBUFFERED"] = "1"
server_environment["WORKER_COUNT"] = "1"
command = [
    str(INFERENCE_SERVER),
    "--entry_script",
    str(SCORE_PATH),
    "--model_dir",
    str(MODEL_DIR),
    "--port",
    str(SCORING_PORT),
    "--worker_count",
    "1",
]

started_at = time.perf_counter()
server_process = subprocess.Popen(
    command,
    cwd=LOCAL_DIR,
    env=server_environment,
    stdout=server_log_handle,
    stderr=subprocess.STDOUT,
)

health_url = f"http://127.0.0.1:{SCORING_PORT}/"
startup_deadline = time.monotonic() + 90
while time.monotonic() < startup_deadline:
    if server_process.poll() is not None:
        server_log_handle.flush()
        raise RuntimeError(SERVER_LOG_PATH.read_text(encoding="utf-8", errors="replace"))
    try:
        health_response = requests.get(health_url, timeout=1)
        if health_response.status_code == 200:
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    stop_local_server()
    raise TimeoutError("The inference server did not become healthy within 90 seconds")

print(f"Server PID: {server_process.pid}")
print(f"Startup seconds: {time.perf_counter() - started_at:.2f}")
print(f"Health: {health_response.status_code} {health_response.text.strip()}")

## 5. Invoke the Scoring Route

A successful response must include the model identity and predictions that match Notebook 01's expected values within the same numerical tolerance.

In [ ]:
score_url = f"http://127.0.0.1:{SCORING_PORT}/score"
request_started_at = time.perf_counter()
score_response = requests.post(score_url, json=request_payload, timeout=30)
request_seconds = time.perf_counter() - request_started_at
score_response.raise_for_status()
response_body = score_response.json()

if response_body["model_name"] != manifest["model_name"]:
    raise AssertionError("Response model name does not match the manifest")
if response_body["model_version"] != manifest["model_version"]:
    raise AssertionError("Response model version does not match the manifest")

predictions = np.asarray(response_body["predictions"], dtype=float)
expected = golden_expected["predict"].to_numpy(dtype=float)
np.testing.assert_allclose(expected, predictions, rtol=1e-6, atol=1e-6)

comparison = pd.DataFrame(
    {
        "expected": expected,
        "local_endpoint": predictions,
        "absolute_error": np.abs(expected - predictions),
    }
)
print(f"Status: {score_response.status_code}")
print(f"Request seconds: {request_seconds:.3f}")
print(f"Maximum absolute error: {comparison['absolute_error'].max():.10f}")
display(comparison.head(10))

## 6. Verify Contract Rejection

A malformed request must fail before H2O prediction. This test removes `pickupHour` from the declared columns and each row.

In [ ]:
invalid_payload = {
    "input_data": {
        "columns": manifest["features"][:-1],
        "data": [row[:-1] for row in request_payload["input_data"]["data"][:1]],
    }
}
invalid_response = requests.post(score_url, json=invalid_payload, timeout=30)
if invalid_response.status_code < 400:
    raise AssertionError("Malformed input was unexpectedly accepted")

print(f"Rejected status: {invalid_response.status_code}")
print(invalid_response.text[:500])

## 7. Inspect Logs and Stop the Server

The final cell always terminates the complete process tree, including the local H2O JVM. Rerun the startup cell to start another test session.

In [ ]:
server_log_handle.flush()
log_lines = SERVER_LOG_PATH.read_text(encoding="utf-8", errors="replace").splitlines()
print("\n".join(log_lines[-40:]))

stop_local_server()
shutdown_deadline = time.monotonic() + 15
while time.monotonic() < shutdown_deadline:
    if not any(port_is_open(port) for port in (SCORING_PORT, 54321)):
        break
    time.sleep(0.5)

open_ports = [port for port in (SCORING_PORT, 54321) if port_is_open(port)]
if open_ports:
    raise RuntimeError(f"Local server ports are still open: {open_ports}")
print("Inference server and H2O stopped cleanly.")

## Result and Azure Handoff

This notebook proves:

- The registered H2O binary artifact can be discovered and hash-verified.
- H2O `3.46.0.12` can start and load the model during Azure ML `init()`.
- The Azure ML inference server accepts the intended JSON contract.
- All 20 golden predictions match exactly.
- Invalid schemas return HTTP 400.
- One-worker process cleanup leaves no local listeners.

It does not create or validate an Azure managed online endpoint. A later cloud deployment should reuse this `score.py` contract with one worker, one initial replica, Microsoft Entra authentication, the prepared endpoint identity, and the exact pinned H2O environment.